[Table of Contents](TOC.ipynb) | [Notebook 02: Data Model ▶](02_data_model.ipynb)

# Notebook 01: Lexical Structure & Basic Execution

Welcome to the first notebook in our bottom-up Python tutorial. This notebook covers the absolute fundamentals: how Python reads, tokens, and compiles code under the hood.

Official Reference: [Python Language Reference - Lexical Analysis](https://docs.python.org/3/reference/lexical_analysis.html)

### A Note on the `print()` Function
Throughout these notebooks, we will use Python's built-in `print()` function to display results. It takes one or more comma-separated arguments, automatically converts them to strings, and outputs them to the console separated by a single space. For example:
-   `print("Hello")` prints a single string.
-   `print("Value is:", 42)` prints the string followed by the integer, separated by a space.

## 1. Physical vs. Logical Lines

In Python, the compilation process begins with a scanner (lexical analyzer) dividing the source code into a sequence of **logical lines**.

*   A **physical line** is a sequence of characters terminated by an end-of-line sequence (LF, CR LF) or EOF.
*   A **logical line** is what the Python interpreter actually executes as a single statement.

This distinction is crucial because Python allows a single logical line to span multiple physical lines using **explicit** or **implicit** line joining.

### Explicit Line Joining
You can join physical lines explicitly using the backslash character (`\`). Crucially:
1.  The backslash must be the last character on that physical line.
2.  It cannot be followed by spaces or comments.

### Implicit Line Joining
Expressions enclosed in parentheses `()`, brackets `[]`, or curly braces `{}` can be split across multiple physical lines without using backslashes. Comments can be appended freely. This is the preferred Pythonic syntax (PEP 8).

In [ ]:
# Explicit line joining (backslash)
logical_line_1 = "This is a long string " \
                  "that spans two physical lines."

# Implicit line joining (preferred PEP 8 style)
logical_line_2 = (
    "This is a long string "
    "using parentheses instead of backslashes. "
    # We can write comments right here!
    "It is cleaner and less error-prone."
)

# Note: Lists (brackets []) and Dictionaries (braces {}) are data structures
# that will be explained in Notebook 06. We use them here to demonstrate that
# brackets and braces also support implicit line joining natively.
implicit_list = [
    "item_1",
    "item_2",  # comments are allowed
    "item_3"
]

implicit_dict = {
    "key_1": "val_1",
    "key_2": "val_2"
}

print(logical_line_1)
print(logical_line_2)
print("Implicit list:", implicit_list)
print("Implicit dict:", implicit_dict)


## 2. Indentation Rules & Blocks

Unlike most modern languages that use curly braces `{}` to define lexical scope or blocks, Python uses leading whitespace (indentation) to group statements.

*   **Syntax Enforced:** All statements within the same block must be indented by the same number of spaces.
*   **Indentation Levels:** Python uses an internal stack to track indentation. When a line starts with more whitespace, it pushes the new indentation level (generating an `INDENT` token). When it returns to a lower indentation level, it pops (generating a `DEDENT` token).
*   **PEP 8 Standard:** Use **4 spaces** per indentation level. Do not mix tabs and spaces.
*   **TabError / IndentationError:** In Python 3, mixing tabs and spaces in the same file is forbidden and raises a `TabError` (subclass of `IndentationError`).

In [ ]:
# Note: try/except catches exceptions (explained in Notebook 08).
# We run the invalid code strings dynamically using exec() so that the notebook
# can run and catch the IndentationError/TabError programmatically.
invalid_code = """
x = 10
  y = 20
"""

try:
    exec(invalid_code)
except IndentationError as e:
    print("Success! Caught IndentationError under the hood:")
    print(e)
    print()

# Mismatched tabs and spaces triggers a TabError (subclass of IndentationError)
mixed_code = "x = 10\n\ty = 20"
try:
    exec(mixed_code)
except TabError as te:
    print("Success! Caught TabError under the hood:")
    print(te)


## 3. Identifiers & Keywords

### Identifiers
An identifier is a name used to identify a variable, function, class, or module. Rules:
1.  Must start with a letter (A-Z, a-z) or an underscore `_`.
2.  Can be followed by letters, underscores, or digits (0-9).
3.  Are case-sensitive (`myVar` != `myvar`).
4.  **Unicode Identifiers:** Under PEP 3131, Python 3 supports Unicode characters in identifiers (e.g. mathematical symbols, Greek letters, non-ASCII characters).

#### Reserved Classes of Identifiers (Underscores)
Python assigns special meaning to certain patterns of leading and trailing underscores in identifiers:
-   **Single Leading Underscore (`_single_leading`):** Indicates a name is intended for internal/private use. For example, `from module import *` will not import names starting with a single underscore.
-   **Double Leading and Trailing Underscores (`__double_leading_and_trailing__`):** Colloquially known as **dunders** (Double UNDERscores). These are system-defined names used by Python to implement core protocols (e.g., `__str__` for string conversion, `__add__` for operator overloading). You should avoid inventing your own dunder names.
-   **Double Leading Underscores (`__double_leading`):** When defined inside a class, the Python compiler performs name mangling (renaming to `_ClassName__identifier`) to prevent name clashes in subclass hierarchies.
-   **Standalone Underscores (`_`, `__`, and `___`):
    *   *Single Underscore (`_`)*: By convention, a single underscore is often used as a "throwaway" variable name to signal that the value is ignored (e.g., `for _ in range(5):` or `x, _ = get_pair()`). In interactive environments (like the Python REPL or Jupyter Notebooks), `_` is also a special system variable that automatically binds to the result of the last evaluated expression.
    *   *Double Underscore (`__`)*: In interactive environments (REPL and Jupyter), `__` automatically holds the result of the second-to-last evaluated expression, and `___` (triple underscore) holds the third-to-last. Outside interactive sessions, some developers also use `__` as a secondary throwaway variable name in nested structures (e.g., nested loops or multi-value unpacking) to avoid name clashes with `_`.

### Keywords
Keywords are reserved words that the parser uses to recognize the structure of the program. They cannot be used as variable names or any other identifier.

### Programmatic Identifier & Keyword Checks
Python provides built-in mechanisms to check whether a string is a valid identifier or keyword:
-   **`str.isidentifier()`:** This string method returns `True` if the string is lexically valid according to the Python language grammar rules. Importantly, because Python keywords are lexically valid identifiers, `"class".isidentifier()` returns `True`!
-   **`keyword` Module:** To check if a name is a reserved keyword, import the `keyword` module and use `keyword.iskeyword(name)`. The full set of keywords in the running Python version can be inspected via the `keyword.kwlist` list.
-   **Valid Identifier Validation:** To determine if a string can actually be bound as a variable name, both checks must be combined: `name.isidentifier() and not keyword.iskeyword(name)`.

In [ ]:
import keyword

# Print all keywords in this Python version
print("Python version has", len(keyword.kwlist), "keywords:")
print(keyword.kwlist)

# Unicode identifier example (PEP 3131)
π = 3.1415926535
r = 5
area = π * (r ** 2)
print()
print("Area of circle with radius", r, "is", area)
print()

# Programmatic identifier checks
print("isidentifier() checks:")
print("'my_var':", 'my_var'.isidentifier())
print("'class':", 'class'.isidentifier())
print("'2vars':", '2vars'.isidentifier())

print()
print("keyword.iskeyword() checks:")
print("'class':", keyword.iskeyword('class'))
print("'my_var':", keyword.iskeyword('my_var'))

# Combined validation check (a variable name must be a valid identifier and not a reserved keyword)
name_1 = "my_var"
is_valid_1 = name_1.isidentifier() and not keyword.iskeyword(name_1)
name_2 = "class"
is_valid_2 = name_2.isidentifier() and not keyword.iskeyword(name_2)

print()
print("Combined variable validation checks:")
print("'my_var' valid variable:", is_valid_1)
print("'class' valid variable:", is_valid_2)


## 4. Literals

Literals are notations for constant values of built-in types.

### Numeric Literals
*   **Integers:** Can be specified in decimal, binary (`0b` or `0B`), octal (`0o` or `0O`), or hexadecimal (`0x` or `0X`).
*   **Underscore Separators:** PEP 515 introduced underscores in numeric literals for readability (e.g., `1_000_000` is parsed exactly as `1000000`).
*   **Floating Point & Complex:** Scientific notation (e.g., `3e-5`) and imaginary numbers (`3j`) are natively supported. Under the hood, Python's `float` type is implemented as a C `double` (IEEE 754 double-precision binary floating-point, using 64 bits). Because computers represent floats in base-2 (binary) rather than base-10 (decimal), many decimal fractions cannot be represented exactly. For example, `0.1` and `0.2` are repeating fractions in binary, leading to representation errors (e.g., `0.1 + 0.2` yields `0.30000000000000004` rather than exactly `0.3`). For applications requiring exact decimal precision (like financial math), use Python's built-in `decimal.Decimal` class.

In [ ]:
# Decimal integer with underscore separator (PEP 515)
decimal_val = 1_000_000
hex_val = 0xFF               # Hexadecimal (255)
bin_val = 0b1010             # Binary (10)
imaginary_val = 2 + 3j       # Complex number

print("Numeric Literals:")
print("Decimal:", decimal_val, "Hex:", hex_val, "Binary:", bin_val, "Complex:", imaginary_val)
print()

# Demonstrating float representation error vs decimal.Decimal:
float_sum = 0.1 + 0.2
print("0.1 + 0.2 yields:", float_sum, "(not exactly 0.3!)")

from decimal import Decimal
decimal_sum = Decimal('0.1') + Decimal('0.2')
print("Decimal('0.1') + Decimal('0.2') yields:", decimal_sum)


### String & Bytes Literals

#### Raw Strings
Raw strings are prefixed with an `r` or `R`. They suppress backslash escaping, meaning that backslashes are treated as literal characters rather than escape sequences. This is highly useful for writing file paths or regular expressions.

In [ ]:
# Raw Strings (suppresses backslash escaping)
raw_path = r"C:\new_folder\test.txt"
print("Raw String:", raw_path)


#### Byte Strings
Byte strings are prefixed with a `b` or `B`. They can contain only ASCII characters or hex escape sequences, representing a sequence of raw 8-bit bytes rather than a Unicode text string (`str`).

In [ ]:
# Byte Strings (represents bytes rather than Unicode str)
byte_data = b"\xff\x00\x7f"
print("Byte String:", byte_data)


#### Formatted Strings
Formatted string literals, or f-strings (introduced in PEP 498), are prefixed with an `f` or `F`. They allow you to embed Python expressions inside string literals using curly braces `{}`. At runtime, these expressions are evaluated and formatted into the string.

In [ ]:
# Formatted Strings (f-strings)
name = "Alice"
print(f"Hello, {name}!")


#### Under the Hood: The Evaluation Replacement Field `{}`
An f-string contains **replacement fields**, which are expressions surrounded by curly braces `{}`.
1.  **Lexical and Syntax Parsing**: During compilation, Python's parser identifies the `f` or `F` prefix and splits the string literal into string constants and replacement fields.
2.  **Runtime Evaluation**:
    *   The expression inside the curly braces `{expression}` is evaluated in the current local/global context.
    *   Python converts the evaluated result to a string. By default, it calls the object's `__str__()` magic method (equivalent to calling `str(value)`).
    *   The resulting string replaces the `{expression}` block in the final output.
3.  **Conversion Flags**: You can specify how the object is converted to a string before formatting using a conversion flag:
    *   `!r`: Calls `repr(value)`, providing a debugging representation (useful for seeing literal quotes/escapes).
    *   `!s`: Calls `str(value)` (the default).
    *   `!a`: Calls `ascii(value)`.
4.  **Escaping Rules**: To include literal curly braces `{` or `}` in an f-string, you must escape them by doubling them: `{{` or `}}`.

In [ ]:
value = "hello\nworld"

# Inline expression evaluation
print(f"Expression evaluation (2 * 3): {2 * 3}")

# Default vs Repr (!r) conversion flags
print(f"Default conversion (str): {value}")
print(f"Repr conversion (!r): {value!r}")
print(f"Ascii conversion (!a): {value!a}")

# Escaping curly braces (doubling braces)
print(f"Literal curly braces: {{this is printed inside braces}}")


#### F-String Formatting Specifiers (`:`)
Python's f-strings support a Format Specification Mini-Language. By placing a colon (`:`) after the expression or conversion flag inside the curly braces, you can provide formatting instructions:
-   **Decimal Precision (`:.Nf`):** Formats floats to `N` decimal places and rounds the value accordingly. For example, `f"{3.14159:.1f}"` evaluates to `"3.1"`.
-   **Zero-Padding (`:0Nd`):** Pads integers with leading zeros to meet a specified total width of `N` characters. For example, `f"{42:06d}"` evaluates to `"000042"`.
-   **Alignment and Width (`:[fill]align[width]`):** Control the layout of the formatted output:
    *   *Width*: An integer specifying the minimum field width (e.g., `{:10}`).
    *   *Align*:
        *   `<`: Left-align within the available space.
        *   `>`: Right-align within the available space.
        *   `^`: Center-align within the available space.
        *   `=`: Pad after the sign (if any) but before the digits.
    *   *Fill*: An optional single character (except `{` or `}`) used to pad the extra space (defaults to space). If specified, it must be placed immediately before the alignment symbol (e.g., `{:*^10}` centered with `*` padding).
-   **CPython Execution Mechanism:** When Python compiles an f-string, it generates bytecode instructions that evaluate the inline expression and then call the built-in `format(value, format_spec)`. This internally invokes `value.__format__(format_spec)`, delegating the formatting logic to the object's type implementation.

In [ ]:
# Float format specifier (precision and rounding)
print(f"Float precision (:.1f): {3.14159:.1f}")

# Integer format specifier (zero-padding to width 6)
print(f"Integer zero-padding (:06d): {42:06d}")

# Alignment and Fill options
print(f"Left-aligned  (:<10):  '{'hello':<10}'")
print(f"Right-aligned (:>10):  '{'hello':>10}'")
print(f"Center-aligned (:^10): '{'hello':^10}'")
print(f"Centered with fill character (:*^10): '{'hello':*^10}'")
print(f"Sign-aware padding (:=+10):           '{42:=+10}'")


#### Alternative String Formatting: Legacy Modulo Operator and Manual Padding
Before PEP 498 introduced f-strings, Python developers used alternative formatting models which are still widely encountered in legacy codebases:
1.  **Legacy Modulo Formatting (`%` operator):** Inherited from C's `printf` syntax, the `%` operator formats values into a format string template containing formatting placeholders (e.g., `%s` for strings, `%d` for integers, `%f` for floating-point numbers). For example, `"%s has %d apples" % ("Alice", 5)` evaluates to `"Alice has 5 apples"`.
2.  **Manual Padding & Alignment Methods:** Python's built-in `str` type provides alignment methods to manually format text output:
    *   `str.rjust(width[, fillchar])`: Returns the string right-aligned in a string of length `width`. Padding is done using the specified `fillchar` (default is space).
    *   `str.ljust(width[, fillchar])`: Returns the string left-aligned.
    *   `str.center(width[, fillchar])`: Returns the string centered.
    *   `str.zfill(width)`: Pads a numeric string with leading zeros. It correctly handles leading signs (e.g., `"-42".zfill(6)` returns `"-00042"`).

In [ ]:
# 1. Modulo Formatting (% operator)
legacy_str = "%s is %d years old." % ("Bob", 30)
print(legacy_str)

# 2. Manual Padding and Alignment methods
print("rjust:".rjust(10), "hello".rjust(10, "*"))
print("ljust:".ljust(10), "hello".ljust(10, "*"))
print("center:".center(10), "hello".center(10, "-"))
print("zfill 6:", "42".zfill(6))
print("zfill -42:", "-42".zfill(6))


---

## Coding Challenges

Complete the exercises below to verify your understanding of Python's lexical structure and basic rules. Run the cell below each challenge to test your solution.

In [ ]:
# ========================================== 
# Challenge 1: Indentation Correction
# ==========================================
# Instructions: The following function contains incorrect and mixed indentation.
# Fix the indentation so that the function compiles and works correctly.
# Ensure that you only use exactly 4 spaces per indentation level.

def calculate_commission(sales_amount, loyalty_years):
# --- FIX THIS CODE BLOCK --- (Adjust indentation levels below)
  if sales_amount <= 0:
    return 0
  else:
    if loyalty_years < 3:
    rate = 0.05  # Indentation error here! Fix me!
    else:
        rate = 0.10
    return sales_amount * rate
# --- END OF FIX BLOCK ---

In [ ]:
# Test assertions for Challenge 1
try:
    assert calculate_commission(1000, 2) == 50.0, "Failed for 1000 sales, 2 years"
    assert calculate_commission(2000, 5) == 200.0, "Failed for 2000 sales, 5 years"
    assert calculate_commission(0, 5) == 0, "Failed for 0 sales"
    print("🎉 Challenge 1 Passed!")
except NameError:
    print("❌ calculate_commission is not defined. Please run the cell above after correcting indentation.")
except AssertionError as e:
    print(f"❌ Challenge 1 Failed: {e}")

In [ ]:
# ==========================================
# Challenge 2: Multiline String Formatting
# ==========================================
# Instructions: Complete the function `generate_report` using a single triple-quoted f-string.
# The function must return a string matching the following EXACT format:
#
# REPORT FOR: <name>
# STATUS: <status>
# RATING: <rating>
# ID: <id_code>
#
# Requirements:
# 1. rating must be formatted to exactly 1 decimal place.
# 2. id_code must be left-padded with zeros to be exactly 6 digits wide (e.g. 42 -> 000042).

def generate_report(name: str, status: str, rating: float, id_code: int) -> str:
    # TODO: Implement this function returning the formatted multiline string.
    pass

In [ ]:
# Test assertions for Challenge 2
try:
    expected_1 = "REPORT FOR: Alice\nSTATUS: Active\nRATING: 4.8\nID: 000123"
    expected_2 = "REPORT FOR: Bob\nSTATUS: Pending\nRATING: 3.0\nID: 000007"
    
    result_1 = generate_report("Alice", "Active", 4.78, 123)
    result_2 = generate_report("Bob", "Pending", 3.01, 7)
    
    assert result_1.strip() == expected_1, f"\nExpected:\n{expected_1!r}\nGot:\n{result_1!r}"
    assert result_2.strip() == expected_2, f"\nExpected:\n{expected_2!r}\nGot:\n{result_2!r}"
    print("🎉 Challenge 2 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 2 Failed: {e}")

In [ ]:
# ==========================================
# Challenge 3: Identifier and Keyword Checker
# ==========================================
# Instructions: Write a function `is_valid_python_variable(name)` that returns True
# if `name` is a valid Python identifier (variable name) and is NOT a reserved keyword.
# Return False otherwise. Hint: check the `keyword` module and string methods.

import keyword

def is_valid_python_variable(name: str) -> bool:
    # TODO: Implement this function.
    pass

In [ ]:
# Test assertions for Challenge 3
try:
    assert is_valid_python_variable("variable_name") == True, "Failed for variable_name"
    assert is_valid_python_variable("_private_var") == True, "Failed for _private_var"
    assert is_valid_python_variable("class") == False, "Failed: class is a keyword!"
    assert is_valid_python_variable("2vars") == False, "Failed: starts with number!"
    assert is_valid_python_variable("valid_unicode_π") == True, "Failed for Unicode support"
    assert is_valid_python_variable("None") == False, "Failed: None is a keyword!"
    assert is_valid_python_variable("none") == True, "Failed: none is not a keyword!"
    print("🎉 Challenge 3 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 3 Failed: {e}")

[Table of Contents](TOC.ipynb) | [Notebook 02: Data Model ▶](02_data_model.ipynb)